# Capstone 1 — Customer Support Resolution Agent · Homework (self-test)

**Goal:** prove you can extend and defend the support agent's design: a new policy rule with tests that catch real bugs, a new handoff field filled by code, error-driven recovery, eval metrics, and one policy adapted to both stacks.

- Theory: [README.md](README.md) (requirements, architecture, acceptance criteria, eval plan, rubric). Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 2–3 hours
- Estimated API cost: **$0**. Everything here is offline.
- Prerequisites: the practice notebook (you do not need its state: this notebook carries its own copy of the policy layer).

**How to use it**
1. Answer the quiz without looking at the solutions at the bottom.
2. Fill in each `TODO` and run its check cell. Checks never crash the notebook; they print a hint when something is off.
3. Write your scenario answer in Part C, then compare it with the rubric.
4. Fill in the self-assessment (Part D) and run the scorecard.

**Scoring:** quiz (10 questions) plus exercises (6) give an overall percentage. The pass mark is **72%**, a self-study bar borrowed from the Claude exams' 720/1000 cut score (the exam score is scaled, so 720 is not literally 72% correct). The self-assessment rubric is reported separately: it tells you whether this capstone is portfolio-ready.

In [ ]:
import copy
import hashlib
import json
import math
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Literal

from agents import ToolGuardrailFunctionOutput
from pydantic import BaseModel, ConfigDict, Field, ValidationError

AUTO_APPROVE_LIMIT_CENTS = 10_000    # $100: refunds up to this are paid at once
HARD_LIMIT_CENTS = 50_000            # $500: refunds above this are never processed by the agent


@dataclass
class SupportState:
    """Per-conversation facts owned by code. account_status and refunded_cents are new in this homework."""
    verified_customer_id: str | None = None
    customer_order_ids: set[str] = field(default_factory=set)
    account_status: str | None = None          # "active" | "suspended", set when get_customer verifies
    refunded_cents: int = 0                    # money already refunded in THIS conversation


@dataclass(frozen=True)
class Decision:
    action: Literal["allow", "ask", "deny"]
    rule: str
    reason: str = ""


def bare(tool_name: str) -> str:
    return tool_name.split("__")[-1]


def decide(tool_name: str, args: dict, state: SupportState) -> Decision:
    """Policy v1 from the practice notebook (rules R1-R4). Do not edit: decide_v2 builds on it."""
    tool = bare(tool_name)
    if tool in ("lookup_order", "process_refund") and state.verified_customer_id is None:
        return Decision("deny", "R1-verify-first",
                        "Identity not verified. Call get_customer with the email on the account first.")
    if tool == "lookup_order" and args.get("order_id") not in state.customer_order_ids:
        return Decision("deny", "R2-own-orders-only", "That order is not on the verified customer's account.")
    if tool == "process_refund":
        if args.get("customer_id") != state.verified_customer_id or args.get("order_id") not in state.customer_order_ids:
            return Decision("deny", "R2-own-orders-only", "Refunds only on the verified customer's own orders.")
        amount = int(args.get("amount_cents") or 0)
        if amount > HARD_LIMIT_CENTS:
            return Decision("deny", "R3-hard-limit",
                            "Refunds above $500 are never processed by the agent. Call escalate_to_human.")
        if amount > AUTO_APPROVE_LIMIT_CENTS:
            return Decision("ask", "R4-supervisor-approval", "Refunds between $100 and $500 need a supervisor.")
    return Decision("allow", "default")


class ActionTaken(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tool: str
    outcome: str


class HandoffSummary(BaseModel):
    """v1 handoff, as in the practice notebook's MCP server."""
    model_config = ConfigDict(extra="forbid")
    escalation_reason: Literal["customer_requested_human", "policy_gap", "threshold_exceeded", "cannot_make_progress"]
    customer_id: str | None
    identity_verified: bool
    order_ids: list[str]
    issue_summary: str
    root_cause: str | None
    refund_amount_cents: int | None
    actions_taken: list[ActionTaken]
    recommended_action: str
    urgency: Literal["low", "normal", "high"]


def prepare_v1(draft: dict, state: SupportState, actions: list[dict]) -> dict:
    """v1 fact-filling: identity and history come from code, never from the model's draft."""
    out = copy.deepcopy(draft)
    out["customer_id"] = state.verified_customer_id
    out["identity_verified"] = state.verified_customer_id is not None
    out["order_ids"] = [o for o in out.get("order_ids", []) if o in state.customer_order_ids]
    out["actions_taken"] = copy.deepcopy(actions)
    return out


def verified(customer_id: str = "C-1001", orders: tuple[str, ...] = ("ORD-5001", "ORD-5002"),
             status: str = "active", refunded: int = 0) -> SupportState:
    """Shortcut for tests: a verified conversation."""
    return SupportState(customer_id, set(orders), status, refunded)


EX_RESULTS: dict[str, bool] = {}
print("policy v1 loaded:", decide("process_refund", {"customer_id": "C-1001", "order_id": "ORD-5001",
                                                      "amount_cents": 4999}, verified()))

## Part A — Quiz

Ten scenario questions about the support agent. Each has exactly one best answer. Write your letters in the cell after question 10.

**Q1.** A verified customer writes: "Just put me through to a human, I don't want to explain this to a bot." The issue sounds like a simple delivery question that the agent could answer with one lookup. What should the agent do?

- A) Look up the order first so the human receives a complete handoff, then escalate
- B) Answer the delivery question, since it is within the agent's capability, and escalate only if the customer asks again
- C) Escalate right away, with a handoff built from what is already known
- D) Run a sentiment classifier and escalate only if frustration crosses a threshold

**Q2.** Refunds sometimes fail because the payment gateway times out. The refund tool currently returns `"Error: refund failed"`, and the agent tells customers that their refund could not be processed. Which change fixes this best?

- A) Add an instruction to the system prompt to retry any failed tool call up to three times before replying
- B) Return a `transient`, `isRetryable: true` error saying no money moved; the agent retries once, then escalates
- C) Return a success result with `refunded: false` and a friendly note, so the agent never alarms the customer
- D) Route every refund-tool failure straight to a human specialist, so no customer hears that a refund failed

**Q3.** After a data migration, `lookup_order` returns `{"found": false}` for every order from the new EU store, because the agent's service account lacks a scope. Customers are told their orders don't exist. What is the best fix?

- A) Have the model confirm the order ID with the customer and only say an order doesn't exist after it is repeated
- B) Retry `lookup_order` with exponential backoff whenever it returns `found: false`, since EU data may still be syncing
- C) State in the system prompt that EU orders are out of scope, so the agent tells EU customers it can't help them
- D) Return a non-retryable `permission` error for unreadable orders and escalate it; keep `found: false` for missing ones

**Q4.** With the OpenAI Agents SDK, refunds above $100 need a supervisor and smaller refunds should run without a pause. The refund tool comes from an MCP server. In the SDK version used in this course, which design implements this?

- A) Set `require_approval={"always": {"tool_names": ["process_refund"]}}` on the MCP server so a supervisor sees each refund
- B) Instruct the agent in its system prompt to ask a supervisor before any refund above $100 and wait for the answer
- C) Wrap the MCP call in a host-side function tool whose `needs_approval` callable reads `amount_cents` from the arguments
- D) Add an output guardrail that checks whether the final message mentions a supervisor's approval for refunds above $100

**Q5.** Supervisors complain that a third of the approval requests they receive are refunds above $500, which the policy forbids the agent to process anyway. What fixes this at the root?

- A) Run the deterministic deny rules before the approval step (`deny` before `ask`), so only approvable refunds reach a person
- B) Give supervisors a one-click "reject everything above $500" button and a checklist that reminds them of the hard limit
- C) Add "never request refunds above $500" to the system prompt, with two examples of large refunds that were escalated
- D) Raise the approval threshold from $100 to $500, so that far fewer refunds create approval requests in the first place

**Q6.** Customers often raise two or three issues in one message. Reviews show that the agent handles the first issue well and silently drops the others in about one in five such tickets, yet the "resolved" rate looks healthy. What is the most effective fix?

- A) Split each incoming message into one ticket per concern, so every concern gets its own conversation and its own reply
- B) Raise `max_tokens` and ask for longer, more detailed replies, so the agent has room to cover every issue it notices
- C) Add "answer every question the customer asks" to the system prompt and keep watching the resolved-rate dashboard
- D) One outcome entry per concern, decompose and investigate before a single reply, and an eval of concerns addressed

**Q7.** After a prompt change, first-contact resolution rose from 76% to 89%, but complaints about wrong refunds also rose. Which pair of metrics is most likely to explain it?

- A) Average tokens per ticket and p95 latency, compared before and after the prompt change on the same traffic
- B) Escalation recall on must-escalate tickets, and policy violations found in the audit log
- C) The model's self-reported confidence on each ticket and the customer's sentiment score at the end of the chat
- D) Accuracy on the stratified test set, recomputed with the new prompt and compared with the previous release

**Q8.** A client wants to cut the cost per ticket. LLM usage costs about $0.02 per ticket; each approval or escalation costs about $4 of staff time; 35% of tickets involve at least one of them. Which first step gives the best evidence-based saving?

- A) Break cost per ticket down by segment, staff time included, and cut the biggest part first, guarded by quality metrics
- B) Move all traffic to the cheapest available model, since the choice of model is the main driver of per-ticket cost
- C) Shorten every tool description and the system prompt to cut input tokens, which are billed again on every turn
- D) Raise the auto-approve limit to $500 so that far fewer refunds need a supervisor, which removes most human touches

**Q9.** A specialist reads a handoff that says "refund of $640 issued", but no refund ran: the policy gate had blocked it. How should the system prevent this?

- A) Instruct the model to re-read the whole conversation and double-check its list of actions before it escalates
- B) Fill identity and `actions_taken` in code from the audit log before filing, and keep the model's narrative fields
- C) Have a second model review every handoff against the transcript before the ticket is filed for the specialist
- D) Remove `actions_taken` from the handoff schema, so the model can no longer make claims about past actions

**Q10.** In your Messages API loop, a response arrives with `stop_reason: "max_tokens"` and a `process_refund` `tool_use` block whose input is missing `reason`. What should the loop do?

- A) Execute the refund, because the order ID and the amount, the fields that move money, are both present
- B) Treat it like `end_turn` and send whatever text the response contains to the customer as the final reply
- C) Don't execute it; retry the turn with a higher `max_tokens`, or end the run through the code-driven escalation
- D) Fill the missing `reason` with `"other"`, which the schema allows, and execute so the customer isn't kept waiting

In [ ]:
answers = {"q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
           "q6": None, "q7": None, "q8": None, "q9": None, "q10": None}   # e.g. "q1": "A"

In [ ]:
ANSWER_KEY = {   # sha256 of "<qid>:<letter>", so the answers are not readable here
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c",
    "q5": "efeafc86344da0b52bbaa7783b6621595e70274062976345ce6c533b9322877c",
    "q6": "70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9",
    "q7": "16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7",
    "q8": "f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc",
    "q9": "b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e",
    "q10": "d6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5",
}


def grade_quiz(answers: dict) -> int:
    score = 0
    for qid, digest in ANSWER_KEY.items():
        letter = (answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        score += ok
        print(f"{qid:>4}: {letter or '-'}  {'✓' if ok else '✗'}")
    print(f"Quiz: {score}/{len(ANSWER_KEY)}")
    return score


QUIZ_SCORE = grade_quiz(answers)

## Part B — Coding exercises

All exercises are offline. Each `check_exN()` prints `✓ Exercise N passed` or a hint.

### Exercise 1 — Add two policy rules

Write `decide_v2()`. It must keep every R1–R4 behavior of `decide()` and add:

- **R5-suspended:** when the verified account's `account_status` is `"suspended"`, deny `lookup_order` and `process_refund` with rule `"R5-suspended"` and a reason that tells the model to call `escalate_to_human`. `get_customer` and `escalate_to_human` stay allowed.
- **R6-session-cap:** a refund that is otherwise allowed must become `ask` (rule `"R6-session-cap"`) when `state.refunded_cents + amount_cents` is **greater than** $100 (`AUTO_APPROVE_LIMIT_CENTS`). This stops a $180 refund from being split into two $90 refunds that each skip approval. Exactly $100 in total is still `allow`.

R1 (verify first) still comes before R5: an unverified conversation has no account status yet.

In [ ]:
def decide_v2(tool_name: str, args: dict, state: SupportState) -> Decision:
    # TODO: R5 and R6 on top of decide()
    raise NotImplementedError

In [ ]:
REFUND = {"customer_id": "C-1001", "order_id": "ORD-5001", "reason": "damaged"}
EX1_CASES = [
    # (label, tool, args, state, expected action, expected rule or None)
    ("R1 still first", "process_refund", {**REFUND, "amount_cents": 4999}, SupportState(), "deny", "R1-verify-first"),
    ("R2 still applies", "lookup_order", {"order_id": "ORD-8001"}, verified(), "deny", "R2-own-orders-only"),
    ("R3 still applies", "process_refund", {**REFUND, "amount_cents": 64000}, verified(), "deny", "R3-hard-limit"),
    ("R4 still applies", "process_refund", {**REFUND, "amount_cents": 25000}, verified(), "ask", None),
    ("plain allow", "process_refund", {**REFUND, "amount_cents": 4999}, verified(), "allow", None),
    ("R5 blocks refunds", "process_refund", {**REFUND, "amount_cents": 4999}, verified(status="suspended"), "deny", "R5-suspended"),
    ("R5 blocks lookups", "lookup_order", {"order_id": "ORD-5001"}, verified(status="suspended"), "deny", "R5-suspended"),
    ("R5 keeps get_customer", "get_customer", {"email": "x@example.com"}, verified(status="suspended"), "allow", None),
    ("R5 keeps escalation", "escalate_to_human", {"handoff": {}}, verified(status="suspended"), "allow", None),
    ("R6 split refund", "process_refund", {**REFUND, "amount_cents": 5000}, verified(refunded=6000), "ask", "R6-session-cap"),
    ("R6 boundary $100 total", "process_refund", {**REFUND, "amount_cents": 4000}, verified(refunded=6000), "allow", None),
    ("R6 with SDK tool name", "mcp__support__process_refund", {**REFUND, "amount_cents": 5000}, verified(refunded=6000), "ask", "R6-session-cap"),
]


def check_ex1() -> bool:
    try:
        failures = []
        for label, tool, args, state, action, rule in EX1_CASES:
            d = decide_v2(tool, args, copy.deepcopy(state))
            if not isinstance(d, Decision):
                failures.append(f"{label}: return a Decision, got {type(d).__name__}")
            elif d.action != action or (rule and d.rule != rule):
                failures.append(f"{label}: expected {action}/{rule or '*'}, got {d.action}/{d.rule}")
            elif rule == "R5-suspended" and "escalate_to_human" not in d.reason:
                failures.append(f"{label}: the reason should tell the model to call escalate_to_human")
        if failures:
            print("✗ Exercise 1:", *failures[:4], sep="\n   ")
            if any("R6" in f for f in failures):
                print("   hint: R6 compares state.refunded_cents + amount_cents with AUTO_APPROVE_LIMIT_CENTS using >, "
                      "and only for refunds decide() would allow.")
            return False
        print("✓ Exercise 1 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 1: decide_v2 is not implemented yet")
    except Exception as exc:
        print(f"✗ Exercise 1: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex1"] = check_ex1()

### Exercise 2 — Tests that catch bugs

A rule is only as good as its tests. Write at least five cases in `MY_POLICY_TESTS`. Each is a dict with `tool`, `args`, `state` and `expect` (`"allow"`, `"ask"` or `"deny"`). The checker runs them against a correct `decide_v2` (every case must pass), checks that at least one case exercises a **new** rule (its expected action differs from v1 `decide()`), and then runs them against three buggy implementations ("mutants"). Your tests must catch at least two of the three. The mutants are hidden, but each one is a plausible mistake in R5 or R6.

In [ ]:
MY_POLICY_TESTS: list[dict] = [
    # TODO, e.g.
    # {"tool": "process_refund", "args": {**REFUND, "amount_cents": 4999}, "state": verified(status="suspended"), "expect": "deny"},
]

In [ ]:
def _reference_v2(tool_name, args, state):
    tool = bare(tool_name)
    if tool in ("lookup_order", "process_refund") and state.verified_customer_id and state.account_status == "suspended":
        return Decision("deny", "R5-suspended", "Account suspended: call escalate_to_human.")
    base = decide(tool_name, args, state)
    if tool == "process_refund" and base.action == "allow" and \
            state.refunded_cents + int(args.get("amount_cents") or 0) > AUTO_APPROVE_LIMIT_CENTS:
        return Decision("ask", "R6-session-cap", "Session total above $100.")
    return base


def _mutant_refund_only(tool_name, args, state):      # R5 forgets lookups
    if bare(tool_name) == "lookup_order":
        return decide(tool_name, args, state)
    return _reference_v2(tool_name, args, state)


def _mutant_no_session(tool_name, args, state):       # R6 ignores earlier refunds
    return _reference_v2(tool_name, args, SupportState(state.verified_customer_id, state.customer_order_ids,
                                                       state.account_status, 0))


def _mutant_boundary(tool_name, args, state):         # R6 uses >= instead of >
    d = _reference_v2(tool_name, args, state)
    if bare(tool_name) == "process_refund" and d.action == "allow" and \
            state.refunded_cents + int(args.get("amount_cents") or 0) >= AUTO_APPROVE_LIMIT_CENTS:
        return Decision("ask", "R6-session-cap", "boundary bug")
    return d


MUTANTS = {"suspended lookups allowed": _mutant_refund_only, "session total ignored": _mutant_no_session,
           "off-by-one at $100": _mutant_boundary}


def _run(policy, case) -> bool:
    return policy(case["tool"], case["args"], copy.deepcopy(case["state"])).action == case["expect"]


def check_ex2() -> bool:
    try:
        tests = list(MY_POLICY_TESTS)
        if len(tests) < 5:
            print(f"✗ Exercise 2: write at least 5 cases (you have {len(tests)})")
            return False
        for i, case in enumerate(tests):
            if not {"tool", "args", "state", "expect"} <= set(case) or case["expect"] not in ("allow", "ask", "deny"):
                print(f"✗ Exercise 2: case {i} needs tool, args, state and expect in allow/ask/deny")
                return False
            if not _run(_reference_v2, case):
                got = _reference_v2(case["tool"], case["args"], copy.deepcopy(case["state"])).action
                print(f"✗ Exercise 2: case {i} expects {case['expect']} but the correct policy says {got}")
                return False
        if not any(decide(c["tool"], c["args"], copy.deepcopy(c["state"])).action != c["expect"] for c in tests):
            print("✗ Exercise 2: no case exercises R5 or R6 (every expectation matches the old policy)")
            return False
        survivors = [name for name, m in MUTANTS.items() if all(_run(m, c) for c in tests)]
        if len(survivors) > 1:
            print(f"✗ Exercise 2: your tests miss these bugs: {', '.join(survivors)}")
            print("   hint: test suspended LOOKUPS, a refund that only crosses $100 together with earlier refunds, "
                  "and the exact $100 boundary.")
            return False
        note = f" (one mutant survived: {survivors[0]})" if survivors else " (all three mutants caught)"
        print("✓ Exercise 2 passed" + note)
        return True
    except Exception as exc:
        print(f"✗ Exercise 2: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex2"] = check_ex2()

### Exercise 3 — Add a handoff field, filled by code

Specialists want to know **when** a reply is due. Add `sla_due_by` to the handoff and fill it in code:

1. `HandoffSummaryV2` extends `HandoffSummary` with a required `sla_due_by: str` field (ISO 8601 UTC, for example `"2026-10-03T14:00:00Z"`). Keep the model closed (`extra="forbid"` is inherited).
2. `finalize_handoff_v2(draft, state, actions, now)` returns a dict that validates against `HandoffSummaryV2`. It applies `prepare_v1()` and sets `sla_due_by = now + SLA_HOURS[urgency]`, formatted as `YYYY-MM-DDTHH:MM:SSZ` in UTC. It **overwrites** any `sla_due_by` the model wrote and must not mutate `draft`.

In [ ]:
SLA_HOURS = {"high": 2, "normal": 8, "low": 24}


class HandoffSummaryV2(HandoffSummary):
    pass   # TODO: add the sla_due_by field


def finalize_handoff_v2(draft: dict, state: SupportState, actions: list[dict], now: datetime) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> bool:
    try:
        schema = HandoffSummaryV2.model_json_schema()
        if "sla_due_by" not in schema.get("properties", {}) or "sla_due_by" not in schema.get("required", []):
            print("✗ Exercise 3: HandoffSummaryV2 needs a required sla_due_by field")
            return False
        if schema.get("additionalProperties") is not False:
            print("✗ Exercise 3: keep the model closed (additionalProperties: false)")
            return False
        now = datetime(2026, 10, 3, 12, 30, tzinfo=timezone.utc)
        draft = {"escalation_reason": "threshold_exceeded", "customer_id": "C-9999", "identity_verified": True,
                 "order_ids": ["ORD-6001", "ORD-0000"], "issue_summary": "Wants $640 back for a failed machine.",
                 "root_cause": None, "refund_amount_cents": 64000,
                 "actions_taken": [{"tool": "process_refund", "outcome": "Refunded $640"}],
                 "recommended_action": "Approve or replace.", "urgency": "high", "sla_due_by": "2099-01-01T00:00:00Z"}
        frozen = copy.deepcopy(draft)
        state = verified("C-1002", ("ORD-6001", "ORD-6002"))
        actions = [{"tool": "process_refund(ORD-6001, 64000)", "outcome": "blocked by R3-hard-limit"}]
        out = finalize_handoff_v2(draft, state, actions, now)
        if draft != frozen:
            print("✗ Exercise 3: finalize_handoff_v2 changed the draft it was given; work on a copy")
            return False
        HandoffSummaryV2.model_validate(out)
        problems = []
        if out["sla_due_by"] != "2026-10-03T14:30:00Z":
            problems.append(f"high urgency at 12:30 UTC should give 2026-10-03T14:30:00Z, got {out['sla_due_by']!r}")
        if out["customer_id"] != "C-1002" or out["order_ids"] != ["ORD-6001"] or out["actions_taken"] != actions:
            problems.append("identity, order_ids and actions_taken must come from prepare_v1 (code), not the draft")
        for urgency, expected in (("normal", "2026-10-03T20:30:00Z"), ("low", "2026-10-04T12:30:00Z")):
            got = finalize_handoff_v2({**frozen, "urgency": urgency}, state, actions, now)["sla_due_by"]
            if got != expected:
                problems.append(f"{urgency}: expected {expected}, got {got!r}")
        if problems:
            print("✗ Exercise 3:", *problems, sep="\n   ")
            return False
        print("✓ Exercise 3 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 3: finalize_handoff_v2 is not implemented yet")
    except ValidationError as exc:
        print(f"✗ Exercise 3: the result does not validate against HandoffSummaryV2: {exc.errors()[0]['loc']} "
              f"{exc.errors()[0]['msg']}")
    except Exception as exc:
        print(f"✗ Exercise 3: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex3"] = check_ex3()

### Exercise 4 — Recover from structured errors

Write `next_action(payload, attempts)`. `payload` is the JSON a tool returned; `attempts` is how many times this exact call has already been made (1 after the first failure). Return one of:

| Situation | Return |
|---|---|
| `payload` has `"blockedBy"` (the policy gate stopped the call) | `"follow_block_reason"` |
| no `errorCategory` (a success, including `found: false`) | `"continue"` |
| `transient` with `isRetryable: true` and `attempts < 2` | `"retry_same_call"` |
| `transient` after that (retry budget used) | `"escalate"` |
| `validation` | `"fix_arguments"` |
| `business` | `"explain_to_customer"` |
| `permission`, or any category you don't recognize | `"escalate"` |

In [ ]:
def next_action(payload: dict, attempts: int) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
EX4_CASES = [
    ({"found": False, "order_id": "ORD-9999"}, 1, "continue"),
    ({"refund_id": "RF-0001", "status": "refunded"}, 1, "continue"),
    ({"blockedBy": "R3-hard-limit", "message": "..."}, 1, "follow_block_reason"),
    ({"errorCategory": "transient", "isRetryable": True, "message": "timeout"}, 1, "retry_same_call"),
    ({"errorCategory": "transient", "isRetryable": True, "message": "timeout"}, 2, "escalate"),
    ({"errorCategory": "transient", "isRetryable": False, "message": "gateway down for maintenance"}, 1, "escalate"),
    ({"errorCategory": "validation", "isRetryable": False, "message": "bad id"}, 1, "fix_arguments"),
    ({"errorCategory": "business", "isRetryable": False, "message": "window", "customerMessage": "..."}, 1, "explain_to_customer"),
    ({"errorCategory": "permission", "isRetryable": False, "message": "scope"}, 1, "escalate"),
    ({"errorCategory": "quota", "isRetryable": True, "message": "new category"}, 1, "escalate"),
]


def check_ex4() -> bool:
    try:
        wrong = [(p, a, exp, next_action(copy.deepcopy(p), a)) for p, a, exp in EX4_CASES]
        wrong = [w for w in wrong if w[2] != w[3]]
        if wrong:
            p, a, exp, got = wrong[0]
            print(f"✗ Exercise 4: {len(wrong)} case(s) wrong; e.g. payload={p} attempts={a}: expected {exp!r}, got {got!r}")
            if p.get("errorCategory") == "transient":
                print("   hint: retry only while isRetryable is true AND the retry budget (attempts < 2) is left.")
            elif "errorCategory" not in p:
                print("   hint: check blockedBy first; a payload without errorCategory is a success, even found=false.")
            return False
        print("✓ Exercise 4 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 4: next_action is not implemented yet")
    except Exception as exc:
        print(f"✗ Exercise 4: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex4"] = check_ex4()

### Exercise 5 — Eval metrics

Each result is a dict with `category`, `expected_status`, `status`, `correct` (bool) and `escalated` (bool: did `escalate_to_human` actually run).

- `escalation_precision_recall(results)` returns `(precision, recall)`, where a positive is `expected_status == "escalated"` and a prediction is `escalated is True`. Return `1.0` for a ratio whose denominator is 0.
- `weighted_fcr(results, mix)` returns the sum over categories of `mix[category] × (share of that category's results with status "resolved" and correct)`. Every category in `mix` appears in `results`.

In [ ]:
def escalation_precision_recall(results: list[dict]) -> tuple[float, float]:
    # TODO
    raise NotImplementedError


def weighted_fcr(results: list[dict], mix: dict[str, float]) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
def _r(category, expected, status, correct, escalated):
    return {"category": category, "expected_status": expected, "status": status, "correct": correct,
            "escalated": escalated}


EX5_RESULTS = [
    _r("order_status", "resolved", "resolved", True, False),
    _r("order_status", "resolved", "resolved", True, False),
    _r("order_status", "resolved", "escalated", False, True),        # over-escalation
    _r("refund", "resolved", "resolved", False, False),              # resolved but wrong amount
    _r("refund", "resolved", "resolved", True, False),
    _r("escalation", "escalated", "escalated", True, True),
    _r("escalation", "escalated", "resolved", False, False),         # missed escalation
]
EX5_MIX = {"order_status": 0.5, "refund": 0.4, "escalation": 0.1}


def _ratio(x) -> bool:
    """A real number in [0, 1]. Rejects NaN and inf, which would slip through abs(x - expected) > tol."""
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) and 0.0 <= x <= 1.0


def check_ex5() -> bool:
    try:
        p, r = escalation_precision_recall(EX5_RESULTS)
        fcr = weighted_fcr(EX5_RESULTS, EX5_MIX)
        if not all(_ratio(x) for x in (p, r, fcr)):
            print(f"✗ Exercise 5: return finite numbers between 0 and 1, got precision={p!r}, recall={r!r}, fcr={fcr!r}")
            return False
        problems = []
        if abs(p - 0.5) > 1e-9 or abs(r - 0.5) > 1e-9:
            problems.append(f"precision/recall should be 0.5/0.5 here, got {p:.3f}/{r:.3f}")
        none = [_r("order_status", "resolved", "resolved", True, False)]
        if escalation_precision_recall(none) != (1.0, 1.0):
            problems.append("with no escalations expected or made, return (1.0, 1.0)")
        expected = 0.5 * (2 / 3) + 0.4 * (1 / 2) + 0.1 * 0
        if abs(fcr - expected) > 1e-9:
            problems.append(f"weighted FCR should be {expected:.4f}, got {fcr:.4f}")
            unweighted = sum(r["status"] == "resolved" and r["correct"] for r in EX5_RESULTS) / len(EX5_RESULTS)
            ignores_correct = 0.5 * (2 / 3) + 0.4 * (2 / 2) + 0.1 * (1 / 2)
            if abs(fcr - unweighted) < 1e-9:
                problems.append("hint: that is an unweighted share; weight each category's rate by mix")
            elif abs(fcr - ignores_correct) < 1e-9:
                problems.append("hint: count a ticket only when status == 'resolved' AND correct is True")
        if problems:
            print("✗ Exercise 5:", *problems, sep="\n   ")
            return False
        print("✓ Exercise 5 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 5: not implemented yet")
    except Exception as exc:
        print(f"✗ Exercise 5: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex5"] = check_ex5()

### Exercise 6 — One policy, two stacks

Write the two adapters that turn a `Decision` into each SDK's native answer:

- `to_pretooluse_output(decision, updated_input=None)` for a Claude Agent SDK `PreToolUse` hook. Always return the envelope `{"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": decision.action, ...}}`. For `deny` and `ask`, add `permissionDecisionReason` set to `decision.reason`. For `allow`, add `updatedInput` only when `updated_input` is given. Why an **explicit** `allow` and not `{}`: in the practice configuration `process_refund` is deliberately left out of `allowed_tools`, and `{}` lets the call continue through the normal permission flow, which would send every in-policy $5 refund to the supervisor's `can_use_tool` callback.
- `to_openai_guardrail(decision)` for an Agents SDK tool input guardrail: `ToolGuardrailFunctionOutput.reject_content(<message>)` for `deny`, where the message contains `decision.reason`; `ToolGuardrailFunctionOutput.allow()` otherwise (`ask` is handled by `needs_approval`, not by the guardrail).

In [ ]:
def to_pretooluse_output(decision: Decision, updated_input: dict | None = None) -> dict:
    # TODO
    raise NotImplementedError


def to_openai_guardrail(decision: Decision) -> ToolGuardrailFunctionOutput:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> bool:
    try:
        deny = Decision("deny", "R3-hard-limit", "Refunds above $500 go to escalate_to_human.")
        ask = Decision("ask", "R4-supervisor-approval", "Needs a supervisor.")
        allow = Decision("allow", "default")
        problems = []
        out = to_pretooluse_output(deny)
        hso = out.get("hookSpecificOutput", {}) if isinstance(out, dict) else {}
        if hso.get("hookEventName") != "PreToolUse" or hso.get("permissionDecision") != "deny" \
                or hso.get("permissionDecisionReason") != deny.reason:
            problems.append(f"deny -> {out}")
        hso = to_pretooluse_output(ask).get("hookSpecificOutput", {})
        if hso.get("permissionDecision") != "ask" or hso.get("permissionDecisionReason") != ask.reason:
            problems.append("ask must produce permissionDecision 'ask' with the reason")
        hso = to_pretooluse_output(allow).get("hookSpecificOutput", {})
        if hso.get("permissionDecision") != "allow" or "updatedInput" in hso:
            problems.append("a plain allow must be an explicit allow (permissionDecision 'allow', no updatedInput); "
                            "{} would hand in-policy refunds to can_use_tool")
        upd = {"handoff": {"customer_id": "C-1001"}}
        hso = to_pretooluse_output(allow, upd).get("hookSpecificOutput", {})
        if hso.get("permissionDecision") != "allow" or hso.get("updatedInput") != upd or "updatedInput" in to_pretooluse_output(allow, upd):
            problems.append("allow + updated_input: updatedInput goes INSIDE hookSpecificOutput with permissionDecision 'allow'")
        g = to_openai_guardrail(deny)
        if not isinstance(g, ToolGuardrailFunctionOutput) or g.behavior.get("type") != "reject_content" \
                or deny.reason not in g.behavior.get("message", ""):
            problems.append("deny -> ToolGuardrailFunctionOutput.reject_content(message containing the reason)")
        for d in (ask, allow):
            g = to_openai_guardrail(d)
            if not isinstance(g, ToolGuardrailFunctionOutput) or g.behavior.get("type") != "allow":
                problems.append(f"{d.action} -> ToolGuardrailFunctionOutput.allow() (approval is needs_approval's job)")
        if problems:
            print("✗ Exercise 6:", *problems, sep="\n   ")
            return False
        print("✓ Exercise 6 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 6: not implemented yet")
    except Exception as exc:
        print(f"✗ Exercise 6: {type(exc).__name__}: {exc}")
    return False


EX_RESULTS["ex6"] = check_ex6()

## Part C — Architecture scenario

Larkspur Home is expanding. Within six months:

1. An **EU storefront** goes live. EU order data must stay in an EU region, and the current orders API cannot read EU orders (today that is the T12 permission error).
2. Legal requires **two independent approvals** for any EU refund above €200.
3. A partner manufacturer offers a **`file_warranty_claim`** API. Product failures after the 30-day window should go there instead of being declined.
4. Chat moves to **three languages** (English, German, Japanese). Specialists only read English.
5. The client's target stays at **≥ 80% first-contact resolution** and **zero policy violations**, and finance wants the cost per ticket (including staff time) to fall by 20%.

Write your design below: what changes in the tools (and their descriptions), the policy layer, each stack (Claude and OpenAI), the handoff, and the eval plan. Name the trade-offs you accept.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Tools.** `file_warranty_claim` is a near neighbour of `process_refund`. Both descriptions gain explicit boundaries (within the window and a refund is wanted → refund; product failure after the window or the customer wants a repair → warranty claim). EU access comes from a separate, region-scoped MCP server or tool deployed in the EU, not a broader scope on the US service account. "Not found" vs "no access" stays distinguishable.
- **Policy layer.** New rules in `decide()`: EU refunds above €200 → a two-approval flow (the second approver must be a different person, enforced in code, not by convention). Currency-aware limits (amounts in minor units plus a currency code; no float money). Warranty claims probably auto-allowed but rate-limited. Every new rule comes with test cases that kill plausible mutants (Exercise 2 style).
- **Claude stack.** The `PreToolUse` gate returns `defer` for the two-approval EU refunds, because the approvals may take hours, and the session resumes after both decisions are recorded. Alternatively, in a Messages API loop, persist the pending `tool_use` and send the `tool_result` later. The MCP server for EU orders is configured as a separate server.
- **OpenAI stack.** `needs_approval` plus a serialized `RunState` stored server-side while approvals are collected. Tool input guardrails still run before approvals. The two-approver rule lives in the review service, which resumes the run only when both approvals are present.
- **Language.** The agent answers in the customer's language, while the handoff stays in English (or carries both): add `customer_language` and an English `issue_summary`. Policy reasons sent to the model stay language-neutral, and the agent phrases them for the customer.
- **Data residency.** EU conversation data, audit logs and handoffs must stay in an EU region. Consider the model provider's regional or data-residency options and say which you would verify with the client's legal team.
- **Eval plan.** The labeled set grows with EU, warranty and German and Japanese tickets, stratified by language. Report FCR per segment and traffic-weighted. Escalation precision and recall per language. Violations from the audit log, including the two-approver rule. Cost per ticket including approvals: the warranty path should *raise* FCR (fewer declines) and the two-approval rule *adds* staff cost, so show the net effect against the 20% goal and propose levers (for example a higher auto-approve limit where the data supports it).
- **Trade-offs named explicitly.** For example: deferring EU refunds lowers first-contact resolution for that segment; a separate EU MCP server adds operational overhead but keeps residency clean.
</details>

## Part D — Self-assessment rubric

This rubric mirrors the one in the [README](README.md#self-assessment-rubric). Criteria marked *auto* are scored from your exercises (2 = all linked exercises pass, 1 = some, 0 = none). Score the others honestly from 0 to 2 in `SELF_ASSESSMENT`: 0 = not done, 1 = done but you could not defend it in an interview, 2 = done and you can explain the trade-offs.

| # | Criterion | Maps to | Scored by |
|---|---|---|---|
| 1 | Policy rules enforced in code, with tests that catch bugs | CCAR-F D1 (1.4, 1.5), ROLE/SAFE/2, ROLE/AGENT/6 | auto: Ex1, Ex2 |
| 2 | Handoff facts filled by code; schema extended safely | CCAR-F D1 (1.4), D5 (5.2) | auto: Ex3 |
| 3 | Structured errors drive recovery (retry, fix, explain, escalate) | CCAR-F D2 (2.2), D5 (5.3 K2) | auto: Ex4 |
| 4 | Eval metrics: weighted FCR, escalation precision and recall | ROLE/EVAL/1, ROLE/EVAL/6 | auto: Ex5 |
| 5 | One policy adapted to both stacks | ROLE/AGENT/7, ROLE/AGENT/8 | auto: Ex6 |
| 6 | Tool descriptions with explicit boundaries for confusable tools | CCAR-F D2 (2.1) | self |
| 7 | Escalation triggers calibrated (explicit request, policy gap, threshold, no progress; not sentiment) | CCAR-F D5 (5.2) | self |
| 8 | Multi-concern requests decomposed and checked | CCAR-F D1 (1.4) | self |
| 9 | Live run reviewed: audit trails read, failure modes noted | ROLE/EVAL/5 | self |
| 10 | Trade-offs explained to a non-technical stakeholder (Part C) | ROLE/COMM/1, ROLE/SAFE/9 | self |

In [ ]:
SELF_ASSESSMENT = {"c6_tool_descriptions": None, "c7_escalation_triggers": None, "c8_multi_concern": None,
                   "c9_live_review": None, "c10_tradeoffs": None}   # each 0, 1 or 2


def rubric_score() -> int:
    def auto(*keys: str) -> int:
        passed = sum(bool(EX_RESULTS.get(k)) for k in keys)
        return 2 if passed == len(keys) else (1 if passed else 0)

    rows = [("1 policy in code + tests", auto("ex1", "ex2")), ("2 handoff field", auto("ex3")),
            ("3 structured errors", auto("ex4")), ("4 eval metrics", auto("ex5")), ("5 both stacks", auto("ex6"))]
    for key, value in SELF_ASSESSMENT.items():
        v = value if value in (0, 1, 2) else 0
        rows.append((key.split("_", 1)[0][1:] + " " + key.split("_", 1)[1].replace("_", " "), v))
    for name, pts in rows:
        print(f"  {name:<28} {pts}/2")
    total = sum(p for _, p in rows)
    verdict = "portfolio-ready" if total >= 16 else "keep iterating"
    print(f"Rubric: {total}/20 -> {verdict} (16+ means you can present this capstone)")
    return total


RUBRIC_TOTAL = rubric_score()

## Scorecard

Run every check cell above first (or use *Run All*).

In [ ]:
QUIZ_SCORE = grade_quiz(answers)
passed = sum(bool(v) for v in EX_RESULTS.values())
overall = (QUIZ_SCORE + passed) / (len(ANSWER_KEY) + 6)
print(f"\nExercises passed: {passed}/6")
print(f"Overall: {overall:.0%} -> {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
print(f"Self-assessment rubric: {rubric_score()}/20")

## Solutions (spoilers)

<details><summary>Quiz answers and explanations</summary>

1. **C.** An explicit request for a person is honored immediately, without first investigating. Hand over what you already know. A delays an explicit request to make the handoff nicer. B is the right pattern for an *upset* customer who has **not** asked for a person. D uses sentiment, which is a weak proxy for case complexity.
2. **B.** The category and the retryable flag let the agent take the one correct action: a bounded retry of the same call, then escalation. A also retries business and validation errors, which never succeed, and retries three times instead of once. C hides a failure as a success. D throws away first-contact resolution for a failure one retry usually fixes.
3. **D.** An access failure that looks like a valid empty result is the most dangerous error shape: the agent confidently tells customers something false. Fix the contract at the source. A and B treat a permission problem as a data or timing problem. C removes the symptom, not the cause, and the EU customers still get no help.
4. **C.** Approval depends on the call's arguments, so the decision needs the arguments. In the installed Agents SDK (0.23.1), an MCP server's callable `require_approval` receives the run context, the agent and the tool definition, not the arguments, and a static list (A) pauses every refund. B is prompt-level and non-deterministic. D runs after the money has moved.
5. **A.** Requests the policy will reject should be rejected by code before a person is involved: deny before ask. B still sends forbidden requests to people, which wastes their time and invites rubber-stamping. C is prompt guidance with a non-zero failure rate. D changes the risk policy without evidence and does nothing about requests above $500.
6. **D.** Decompose, investigate (in parallel where the lookups are independent), answer once, and make the decomposition **measurable**. A loses the shared context and sends the customer three replies. B does not address the cause. C alone is unenforced guidance with nothing that detects a regression.
7. **B.** A jump in FCR together with more wrong refunds usually means the agent stopped escalating cases it should have escalated and started acting outside the policy. Escalation recall and audit-log violations show exactly that. A, C and D cannot tell a correct resolution from a confident wrong one.
8. **A.** At $4 per human touch against $0.02 of model usage, staff time dominates. Measure by segment first, then cut the biggest component, guarded by the quality metrics. B saves pennies and risks quality. C hurts tool selection to save a negligible amount. D changes the risk policy without evidence.
9. **B.** Facts that code knows (who was verified, what ran, what was blocked) should be filled by code. The model writes the narrative. A and C are probabilistic checks of something that can be made deterministic. D removes information the specialist needs.
10. **C.** A response cut off by `max_tokens` may contain an incomplete `tool_use`. Never execute it; retry with more room, or end the run into the code-driven escalation path. A and D act on a truncated instruction, and D invents a value the customer never gave. B confuses truncation with completion.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
# reference: ex1
def decide_v2(tool_name: str, args: dict, state: SupportState) -> Decision:
    tool = bare(tool_name)
    if (tool in ("lookup_order", "process_refund") and state.verified_customer_id is not None
            and state.account_status == "suspended"):
        return Decision("deny", "R5-suspended",
                        "This account is suspended. Do not look up orders or refund. Call escalate_to_human "
                        "with escalation_reason='policy_gap' and explain that a specialist will follow up.")
    base = decide(tool_name, args, state)                     # R1-R4 unchanged (R1 still comes first)
    if tool == "process_refund" and base.action == "allow":
        amount = int(args.get("amount_cents") or 0)
        if state.refunded_cents + amount > AUTO_APPROVE_LIMIT_CENTS:
            return Decision("ask", "R6-session-cap",
                            "Refunds in this conversation would exceed $100 in total, so a supervisor must approve.")
    return base
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
# reference: ex2
MY_POLICY_TESTS = [
    {"tool": "process_refund", "args": {**REFUND, "amount_cents": 4999}, "state": verified(status="suspended"), "expect": "deny"},
    {"tool": "lookup_order", "args": {"order_id": "ORD-5001"}, "state": verified(status="suspended"), "expect": "deny"},
    {"tool": "get_customer", "args": {"email": "jane@example.com"}, "state": verified(status="suspended"), "expect": "allow"},
    {"tool": "process_refund", "args": {**REFUND, "amount_cents": 5000}, "state": verified(refunded=6000), "expect": "ask"},
    {"tool": "process_refund", "args": {**REFUND, "amount_cents": 4000}, "state": verified(refunded=6000), "expect": "allow"},
    {"tool": "process_refund", "args": {**REFUND, "amount_cents": 4999}, "state": SupportState(), "expect": "deny"},
]
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
# reference: ex3
class HandoffSummaryV2(HandoffSummary):
    sla_due_by: str = Field(description="Reply due by, ISO 8601 UTC, e.g. 2026-10-03T14:00:00Z. Set by code.")


def finalize_handoff_v2(draft: dict, state: SupportState, actions: list[dict], now: datetime) -> dict:
    out = prepare_v1(draft, state, actions)                   # prepare_v1 already works on a deep copy
    due = now.astimezone(timezone.utc) + timedelta(hours=SLA_HOURS[out["urgency"]])
    out["sla_due_by"] = due.strftime("%Y-%m-%dT%H:%M:%SZ")
    return out
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
# reference: ex4
def next_action(payload: dict, attempts: int) -> str:
    if "blockedBy" in payload:
        return "follow_block_reason"
    category = payload.get("errorCategory")
    if category is None:
        return "continue"
    if category == "transient":
        return "retry_same_call" if payload.get("isRetryable") and attempts < 2 else "escalate"
    return {"validation": "fix_arguments", "business": "explain_to_customer"}.get(category, "escalate")
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
# reference: ex5
def escalation_precision_recall(results: list[dict]) -> tuple[float, float]:
    should = [r for r in results if r["expected_status"] == "escalated"]
    did = [r for r in results if r["escalated"]]
    hits = sum(1 for r in did if r["expected_status"] == "escalated")
    precision = hits / len(did) if did else 1.0
    recall = hits / len(should) if should else 1.0
    return precision, recall


def weighted_fcr(results: list[dict], mix: dict[str, float]) -> float:
    total = 0.0
    for category, weight in mix.items():
        rows = [r for r in results if r["category"] == category]
        resolved = sum(1 for r in rows if r["status"] == "resolved" and r["correct"])
        total += weight * resolved / len(rows)
    return total
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
# reference: ex6
def to_pretooluse_output(decision: Decision, updated_input: dict | None = None) -> dict:
    out = {"hookEventName": "PreToolUse", "permissionDecision": decision.action}
    if decision.action == "allow":
        if updated_input is not None:
            out["updatedInput"] = updated_input
    else:
        out["permissionDecisionReason"] = decision.reason
    return {"hookSpecificOutput": out}


def to_openai_guardrail(decision: Decision) -> ToolGuardrailFunctionOutput:
    if decision.action == "deny":
        return ToolGuardrailFunctionOutput.reject_content(
            json.dumps({"blockedBy": decision.rule, "message": decision.reason}))
    return ToolGuardrailFunctionOutput.allow()
```
</details>

Next: [Capstone 2 — Multi-agent research system with provenance](../02-multi-agent-research-with-provenance/README.md). It builds on [Chapter 10 — Multi-agent error handling](../../../anthropic-claude/10-multi-agent-error-handling/README.md) and [Chapter 12 — Provenance](../../../anthropic-claude/12-provenance/README.md).